# The Language Gap: analysis demo

This notebook reproduces the results in Report 4 from the files saved in the GitHub repository: it scores the saved chatbot replies, checks the scores against my hand check, and runs the statistical analysis and figures. No API key is needed, because the chatbot replies are already in `data/responses/`.

**How to run:** Runtime → Run all. It takes about a minute.

Repository: https://github.com/ammarhafiy/ds440-language-gap

In [ ]:
!git clone -q https://github.com/ammarhafiy/ds440-language-gap.git
%cd ds440-language-gap
!pip install -q -r requirements.txt

## 1. Score the replies (script 06)

Each reply is matched against the accepted answers (rules in `notes/scoring_rubric.md`). 8,390 replies per chatbot.

In [ ]:
!python scripts/06_score.py --files data/responses/gemini_gemini-3.1-flash-lite.csv data/responses/sealion_Qwen-SEA-LION-v4.5-27B-IT.csv

## 2. Check the automatic scores against my hand check (script 07)

I scored 200 random replies per chatbot by hand, blind to the automatic scores. Cohen's kappa measures the agreement.

In [ ]:
!python scripts/07_hand_check.py kappa --scored data/scored/gemini_gemini-3.1-flash-lite_scored.csv
!python scripts/07_hand_check.py kappa --scored data/scored/sealion_Qwen-SEA-LION-v4.5-27B-IT_scored.csv

## 3. Statistical analysis and figures (script 08)

Accuracy with 95% confidence intervals, McNemar's test (H1), the English–Malay gap by league (H2) and with the fact (H3), the Malay Wikipedia regression (H4), and two sensitivity checks.

In [ ]:
!python scripts/08_analysis.py

In [ ]:
from IPython.display import Image, display
for name in ["fig1_accuracy_no_fact", "fig2_fact_effect", "fig3_malay_wikipedia"]:
    display(Image(f"results/analysis/figures/{name}.png"))

In [ ]:
import pandas as pd
pd.read_csv("results/analysis/table2_h1_mcnemar.csv")

## 4. Try the scoring yourself

Change the reply below and run the cell to see whether the automatic scoring counts it as correct. The example question is a Malaysia Super League player born in Marang.

In [ ]:
import importlib.util
spec = importlib.util.spec_from_file_location("score", "scripts/06_score.py")
score = importlib.util.module_from_spec(spec)
spec.loader.exec_module(score)
questions = {q["question_id"]: q for q in score.read_csv(score.QUESTIONS_PATH)}

q = questions["msl-Q6894057-P1"]
print("Question (English):", q["question_en"])
print("Question (Malay):  ", q["question_ms"])
print("Accepted answers:  ", q["accepted_answers"])
print()
for reply in ["Marang, Terengganu.", "Kuala Terengganu, Malaysia.", "Beliau dilahirkan di Pekan Marang."]:
    correct, matched = score.score_reply(reply, score.accepted_forms(q))
    print(f"{reply!r:45} -> {'correct' if correct else 'incorrect'}")